<a href="https://colab.research.google.com/github/Tuchobm/Curso-IA-Google-Colab/blob/main/3_3_generacion_datos_result.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Ejercicio 3: Aprendizaje no Supervisado - Generación de Características

En este ejercicio se aplican técnicas de generación de características a un conjunto de datos de imágenes. El objetivo es usando datos sintéticos pueda ayudar a mejorar el rendimiento de un modelo de clasificación.

Además, encontrarás partes del código contendrán  el comentario de `# ACTIVIDAD` que indican dónde debes completar el código o realizar tareas específicas. Asegúrate de seguir las instrucciones y completar el código donde se indique. Finalmente, en el final de este ejercicio, deberás responder a una serie de preguntas.

In [ ]:
from sklearn.model_selection import train_test_split
from sklearn.datasets import load_digits
import pandas as pd
import numpy as np


def load_data():
    """
    Carga el conjunto de datos de dígitos escritos a mano y lo divide en
    conjuntos de entrenamiento y prueba.

    Returns:
        tuple: (X_train, X_test, y_train, y_test) donde X son las características
        y y son las etiquetas.
    """
    # Cargar el conjunto de datos
    data = load_digits()

    # Crear un DataFrame con las características y etiquetas
    df = pd.DataFrame(data.data, columns=data.feature_names)
    df.insert(0, "target", data.target)

    # Dividir el conjunto de datos en entrenamiento y prueba
    entrenamiento, prueba = train_test_split(df, test_size=0.99, random_state=0)

    return entrenamiento, prueba

# Entrenamiento y evaluación de un modelo de clasificación

In [ ]:
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.svm import SVC

pipe = Pipeline(
    [
        ("scaler", StandardScaler()),
        ("model", SVC(random_state=0)),
    ]
)

# Generación de datos sintéticos

Ideas para la generación de datos sintéticos:
- Mover la imagen en una dirección una cierta cantidad de píxeles (ndimage.shift)
- Rotar la imagen (ndimage.rotate)
- Aumentar el brillo de la imagen (np.clip)
- Aumentar el contraste de la imagen [`(img - min_val) * (16.0 / (max_val - min_val)`]
- Reducir o aumentar el tamaño de la imagen (ndimage.zoom)
- Deformar la imagen con ruido gaussiano (ndimage.gaussian_filter)
- Combinar técnicas de generación de características (por ejemplo, rotar y mover la imagen)
- (Complejo) Entrenar un modelo GAN para generar imágenes similares a las de la base de datos (e.g. usar librería ydata-synthetic en versión 1.4.0)

<!--
!pip install ydata-synthetic==1.4.0
from ydata_synthetic.synthesizers.regular import RegularSynthesizer
from ydata_synthetic.synthesizers.base import ModelParameters, TrainParameters

ctgan_args = ModelParameters(
    batch_size=500,
    lr=1e-4,
    betas=(0.9, 0.99),
)

train_args = TrainParameters(epochs=100)

synth = RegularSynthesizer(modelname="wgangp", model_parameters=ctgan_args)
synth.fit(
    data=entrenamiento,
    train_arguments=train_args,
    num_cols=entrenamiento.columns[1:].tolist(),
    cat_cols=["target"],
)

data = synth.sample(1)

img = data.iloc[0]
print(img["target"])
img = img[1:].values.reshape(8, 8)
-->
(Haz doble clic en la celda para una pista de cómo hacerla ultima opción)

In [ ]:
from scipy import ndimage

entrenamiento, prueba = load_data()

# Generación de datos sinteticos
datos = [entrenamiento]
images = entrenamiento.iloc[:, 1:].values.reshape(-1, 8, 8)

# Ejemplo: movimiento de píxeles a la derecha
movido = entrenamiento.copy()
movido.iloc[:, 1:] = np.vstack(
    [ndimage.shift(image, (0, 1)).reshape(1, -1) for image in images]
)
datos.append(movido)

# ACTIVIDAD: Generar 3 transformaciones adicionales
# rotado = entrenamiento.copy()
# rotado.iloc[:, 1:] = np.vstack(
#     [ndimage.rotate(image, 45, reshape=False).reshape(1, -1) for image in images]
# )
# datos.append(rotado)

# ruido = entrenamiento.copy()
# ruido.iloc[:, 1:] = np.vstack(
#     [ndimage.gaussian_filter(image, sigma=1).reshape(1, -1) for image in images]
# )
# datos.append(ruido)

# iluminado = entrenamiento.copy()
# iluminado.iloc[:, 1:] = np.vstack(
#     [np.clip(image + 0.5, 0, 16).reshape(1, -1) for image in images]
# )
# datos.append(iluminado)

entrenamiento = pd.concat(datos, ignore_index=True)

In [ ]:
pipe.fit(entrenamiento.drop(columns="target"), entrenamiento["target"])

Pipeline(steps=[('scaler', StandardScaler()), ('model', SVC(random_state=0))])

In [ ]:
from sklearn.metrics import accuracy_score


def score_pred(df):
    X = df.drop(columns="target")
    y_pred = pipe.predict(X)
    print(f"Acuraccy: {accuracy_score(df['target'], y_pred):.4f}")


score_pred(entrenamiento)
score_pred(prueba)

Acuraccy: 1.0000
Acuraccy: 0.4556


# Datos imbalancedos

Segunda parte del ejercicio es la generación de datos sintéticos para balancear un conjunto de datos imbalancedos. Donde una categoría tiene muchas más muestras que otra.

Utilizaremos la librería `imblearn` para generar datos para balancear el conjunto de datos. En este caso, utilizaremos la técnica de sobremuestreo SMOTE (Synthetic Minority Over-sampling Technique) para generar datos sintéticos para la clase minoritaria.

In [ ]:
from sklearn.datasets import make_classification
from sklearn.model_selection import train_test_split

X, y = make_classification(
    n_samples=1000,
    n_features=2,
    n_informative=2,
    n_redundant=0,
    n_classes=2,
    weights=[0.02, 0.98],
    class_sep=0.5,
    flip_y=0.01,
    random_state=0,
)

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.3, random_state=0, stratify=y
)

In [ ]:
from imblearn.over_sampling import SMOTE

ros = SMOTE(random_state=0)
X_resampled, y_resampled = ros.fit_resample(X_train, y_train)

In [ ]:
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import confusion_matrix

# ACTIVIDAD: Entrenar dos modelos RandomForestClassifier, uno con los datos originales y otro con los datos balanceados
model = RandomForestClassifier(random_state=0)
model.fit(X_train, y_train)
y_pred = model.predict(X_test)
print(confusion_matrix(y_test, y_pred))

model = RandomForestClassifier(random_state=0)
model.fit(X_resampled, y_resampled)
y_pred = model.predict(X_test)
print(confusion_matrix(y_test, y_pred))

[[  0   7]
 [  2 291]]
[[  6   1]
 [ 37 256]]


# Preguntas

- ¿Qué efecto tiene en la accuracy las técnicas de generación de datos sintéticos en el rendimiento del modelo? (Prueba por lo menos 3 técnicas diferentes)
  * Sin generación sintética: 0.3292
  * Movido a la derecha: 0.4466
  * Rotado: 0.3073
  * Ruido: 0.4225
  * Iluminación: 0.4556

- ¿Se te ocurre alguna técnica de generación de datos sintéticos más?
  * Eliminar el ruido de la imagen.
  * Generar imágenes de fondo para que el modelo sea más robusto a diferentes escenarios.
  * Invertir horizontalmente o verticalmente la imagen.
- ¿Qué problema tiene un modelo de clasificación si no se entrena con datos balanceados? ¿Qué rendimiento tiene respecto a las clases minoritarias?
  * Generalmente el modelo no aprende a clasificar correctamente las clases minoritarias, por lo que su rendimiento es muy bajo para estas clases. En este caso, el modelo sin datos balanceados no acertó ninguna clase 0, mientras que usando SMOTE, el modelo acertó 6 de 7 en la clase 0.
- ¿Qué efecto tiene el uso de SMOTE en el rendimiento del modelo? ¿Por qué?
  * El uso de SMOTE mejora el rendimiento del modelo, disminuyendo el número de falsos clasificaciones minoritarias, pero disminuye el rendimiento en la clase mayoritaria. Esto se debe a que el modelo tiene más ejemplos de la clase minoritaria para aprender, pero también puede llevar a un sobreajuste si no se usa correctamente.
- ¿Qué modelo de clasificación es más robusto a datos imbalancedos? ¿Por qué?
  * Random Forest o AdaBoost son más robustos a datos imbalancedos, ya que son modelos de ensamblado que combinan múltiples árboles de decisión, lo que les permite aprender patrones más complejos y generalizar mejor a datos imbalanceados. Esto se debe a que cada árbol de decisión se entrena en una muestra aleatoria de los datos, lo que les permite aprender patrones diferentes y tiene más oportunidad de aprender de las clases minoritarias. Además, estos modelos pueden manejar datos ruidosos y no lineales, lo que los hace más robustos a datos imbalancedos.